In [29]:
from typing import NamedTuple, TypedDict, AnyStr, Callable, Any, TypeVar

from librosa import LibrosaError
from pedalboard import
import pandas as pd
import requests
from pathlib import Path

from narwhals import DataFrame

from tqdm.notebook import tqdm
import zipfile
import librosa
import IPython
from glob import glob
import numpy as np

In [30]:
np.random.seed(0)

In [31]:
# https://stackoverflow.com/a/62113293
def download(url, path: Path):
    resp = requests.get(url, stream=True)
    total = int(resp.headers.get('content-length', 0))
    with open(path, 'wb') as file, tqdm(
            desc=path.name,
            total=total,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
    ) as bar:
        for data in resp.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

In [32]:
download_path = Path("data/dataset.zip")
if not (download_path.exists() and download_path.is_file()):
    download("https://zenodo.org/records/3371780/files/audio_mono-mic.zip?download=1", download_path)

In [33]:
guitar_set_dir = Path("data/GuitarSet")
if not (guitar_set_dir.exists() and guitar_set_dir.is_dir()):
    with zipfile.ZipFile(download_path, "r") as zip_ref:
        for member in tqdm(zip_ref.infolist(), desc="Extracting "):
            zip_ref.extract(member, guitar_set_dir)

In [34]:
# example, sr = librosa.load(guitar_set_dir.joinpath("00_BN1-129-Eb_comp_mic.wav"))
# IPython.display.Audio(data = example, rate=sr)

In [35]:
class Sample(NamedTuple):
    audio: Path
    x: np.ndarray
    sr: int
    sample_start: int
    sample_end: int



class SampleWithFeatures(NamedTuple):
    audio: Path
    x: np.ndarray
    sr: int
    sample_start: int
    sample_end: int
    zero_crossing_rate: float
    spectral_centroid: float
    rms: float

In [36]:
T = TypeVar('T')
def split_into_samples(files: list[AnyStr], sample_length: int, f: Callable[[Sample], T]) -> list[T]:
    dataset_list: list[T] = []
    for file in files:
        try:
            sample, sr = librosa.load(Path(file))
        except LibrosaError as e:
            print(f"Failed to load {file}: {e}")
            continue
        length: float = len(sample) / sr
        sample_no = int(length // sample_length)
        remainder: float = length - sample_length * sample_no

        last_offset:int = 0

        for i in range(sample_no):
            offset: float = np.random.random()*remainder
            remainder -= max(0, offset)
            start_index = last_offset + int(offset*sr)
            end_index = min(int(last_offset + offset*sr + sample_length*sr), len(sample))
            dataset_list.append(f(Sample(audio=Path(file), x=sample, sr=sr, sample_start=start_index, sample_end=end_index)))
            last_offset = end_index
    return dataset_list

In [37]:
files = glob(str(guitar_set_dir)+"/*")

In [38]:
dataset_5sec: DataFrame = pd.DataFrame(split_into_samples(files, 5, lambda x: x))
display(dataset_5sec[dataset_5sec["audio"].apply(str)=="data/GuitarSet/00_SS3-84-Bb_comp_mic.wav"])
dataset_10sec: DataFrame = pd.DataFrame(split_into_samples(files, 10, lambda x: x))
display(dataset_10sec[dataset_10sec["audio"].apply(str)=="data/GuitarSet/00_SS3-84-Bb_comp_mic.wav"])

,audio,x,sr,sample_start,sample_end
0,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,8563,118813
1,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,123847,234097
2,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,235305,345555
3,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,345988,456238
4,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,456391,566641
5,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,566775,677025
6,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,677057,787307
7,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,787344,897594
8,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,897598,1007848


,audio,x,sr,sample_start,sample_end
0,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,76146,296646
1,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,311702,532202
2,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,552204,772704
3,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,"[-0.000045, 0.000043, 0.000121, 0.000109, 0.00...",22050,775189,995689


In [39]:
def get_audio(df: DataFrame, i: int)->tuple[np.ndarray, int]:
    row = df.iloc[i]
    x, sr = librosa.load(row.audio)
    return x[row.sample_start:row.sample_end], sr

In [40]:
x, sr = get_audio(dataset_10sec, 0)
IPython.display.Audio(x, rate=sr)

In [42]:
x, sr = get_audio(dataset_5sec, 1)
IPython.display.Audio(x, rate=sr)